# Intentos de suicidio notificados en Medellín: diferencias territoriales y temporales

**Hackathon de Datos ITM 2026** · Modalidad: *Análisis de datos* · Temática: *Salud y bienestar*

**Pregunta central:** ¿Qué diferencias territoriales y temporales presentan las tasas de intentos de suicidio notificados en Medellín, y cómo utilizar estos indicadores para apoyar una priorización exploratoria de acciones de prevención y bienestar?

**Fuentes (MEData, Alcaldía de Medellín):**
1. [Intento de suicidio (SIVIGILA)](https://medata.app.medellin.gov.co/dataset/1-026-22-000146/): casos notificados, 2014–2022.
2. [Mortalidad](https://medata.app.medellin.gov.co/dataset/1-026-22-000161/): defunciones con causa básica CIE-10, 2012–2021.
3. [Alertas recibidas por el Sistema de Alertas Tempranas de Medellín (SALVAVIDAS)](https://medata.app.medellin.gov.co/dataset/1-020-13-000422/): Alertas individuales recibidas por el Sistema de Alertas Tempranas de Medellín (SALVAVIDAS).
4. [Factores de riesgo identificados por el Sistema de Alertas Tempranas de Medellín (SALVAVIDAS)](https://medata.app.medellin.gov.co/dataset/1-020-13-000421/) Factores de riesgo identificados por el Sistema de Alertas Tempranas de Medellín (SALVAVIDAS).
5. [Alertas asociadas a factores de riesgo identificados por el Sistema de Alertas Tempranas de Medellín (SALVAVIDAS)](https://medata.app.medellin.gov.co/dataset/1-020-13-000423/) Alertas asociadas a factores de riesgo.
6. [Proyecciones de población por comuna 2018–2030](https://www.medellin.gov.co/es/centro-documental/proyecciones-poblacion-viviendas-y-hogares/) (DAP con DANE, actualización 2025), para calcular tasas.
> Nota ética: se trabaja solo con datos agregados (conteos y tasas). No se identifica a ninguna persona.
> Si este tema te toca de cerca, busca apoyo en tu EPS o en las líneas de salud mental de la Alcaldía de Medellín. En una emergencia, llama al 123.

## 1. Configuración

Para ejecutar en Google Colab: sube los archivos CSV (y el Excel de población, si lo tienes) al panel de archivos, o deja que la celda descargue la base de intentos directamente de MEData.

In [ ]:
# dependencias
%pip install pandas numpy matplotlib seaborn scikit-learn scipy openpyxl folium

In [ ]:
import io
import time
import random
import http.client
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#Configuracion por defecto de tablas
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})

UA = {"User-Agent": "Mozilla/5.0 (scraper hackathon ITM) "}
TIMEOUT = 120
REINTENTOS = 4
CARPETA_DATOS = '.'
CODIGOS_REINTENTABLES = {408, 429, 500, 502, 503, 504}


URL_SIVIGILA = "https://medata.app.medellin.gov.co/sites/default/files/resources/fef722ec5c04d8b9572c3d82000c08fc_1743010212/sivigila_intsuicidio.csv" 
URL_AFECTADOS_ALERTAS_SATMED = 'https://medata.app.medellin.gov.co/sites/default/files/resources/85ae226483faf359ef0836766c8cf1f4_1743010380/afectados_alertas_recibidas_satmed.csv'
URL_FACTORES_RIESGO_SATMED = 'https://medata.app.medellin.gov.co/sites/default/files/resources/47bead27a14db377f5490b78ea8e5b05_1743010383/factores_riesgos_identificados_satmed.csv'
URL_ALERTAS_FACTORES_RIESGO_SATMED = 'https://medata.app.medellin.gov.co/sites/default/files/resources/419d71ce415acfc78fb4e63a1173e837_1743010381/alertas_asociadas_factores_riesgos_identificados_satmed.csv'
URL_DEFUNCIONES = 'https://medata.app.medellin.gov.co/sites/default/files/resources/3ffc1b499f9fd464ed0b8434ae38d775_1743010223/defunciones.csv'
URL_CENCO_PREDICCIONES_PROYECCIONES_POR_COMUNA = ("https://www.medellin.gov.co/es/wp-content/uploads/2022/08/"
                    "1.-Proyecciones-Medellin-por-Comunas-y-Corregimientos-2018-2030-Actualizado.xlsx") 


URLS = {
    "sivigila": URL_SIVIGILA,
    "afectados_alertas": URL_AFECTADOS_ALERTAS_SATMED,
    "factores_riesgo": URL_FACTORES_RIESGO_SATMED,
    "alertas_factores_riesgo": URL_ALERTAS_FACTORES_RIESGO_SATMED,
    "defunciones": URL_DEFUNCIONES
}

KWARGS_PROYECCIONES = {"header": 15, "dtype": {"Código Comuna": str}}


COMUNAS = {
    "01": "Popular", "02": "Santa Cruz", "03": "Manrique", "04": "Aranjuez", "05": "Castilla",
    "06": "Doce de Octubre", "07": "Robledo", "08": "Villa Hermosa", "09": "Buenos Aires",
    "10": "La Candelaria", "11": "Laureles", "12": "La América", "13": "San Javier",
    "14": "El Poblado", "15": "Guayabal", "16": "Belén",
    "50": "Palmitas", "60": "San Cristóbal", "70": "Altavista", "80": "San Antonio de Prado",
    "90": "Santa Elena",
}
AZUL, GRIS, ROJO = "#2f5d8a", "#b9c0c8", "#b8463b"

class ContenidoInvalido(Exception):
    pass

def _es_reintentable(e):
    if isinstance(e, urllib.error.HTTPError):
        return e.code in CODIGOS_REINTENTABLES
    return isinstance(e, (OSError, http.client.HTTPException, ContenidoInvalido))


def _get(url, timeout=TIMEOUT, reintentos=REINTENTOS):
    ultimo = None
    for i in range(reintentos):
        try:
            req = urllib.request.Request(url, headers=UA)
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read()
            if datos.lstrip()[:15].lower().startswith((b"<!doctype", b"<html")):
                raise ContenidoInvalido("devolvió HTML en vez de CSV")
            return datos        
        except Exception as e:
            ultimo = e
            if not _es_reintentable(e):
                break
            if i < reintentos - 1:
                time.sleep(2 ** i + random.random())
    raise RuntimeError(f"GET {url} falló: {ultimo!r}") from ultimo

def leer_csv(url, **kwargs):
    kwargs.setdefault("low_memory", False)
    return pd.read_csv(io.BytesIO(_get(url)), **kwargs)

def _urls_validas(solo=None):
    validas = {}
    for nombre, url in URLS.items():
        if solo and nombre not in solo:
            continue
        if url:
            validas[nombre] = url
        else:
            print(f"saltado (sin URL): {nombre}")
    return validas


def _cargar_uno(url, carpeta, forzar, kwargs):
    if not carpeta:                      # solo memoria
        return leer_csv(url, **kwargs)

    ruta = Path(carpeta) / url.split("/")[-1].split("?")[0]
    if forzar or not ruta.exists():
        ruta.parent.mkdir(parents=True, exist_ok=True)
        parcial = ruta.with_suffix(ruta.suffix + ".part")
        parcial.write_bytes(_get(url))
        parcial.replace(ruta)            # renombrado atómico
    return pd.read_csv(ruta, **kwargs)

def cargar_todo(carpeta=None, forzar=False, workers=4, solo=None, **kwargs):
    kwargs.setdefault("low_memory", False)
    dfs, fallos = {}, {}

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futuros = {
            ex.submit(_cargar_uno, url, carpeta, forzar, kwargs): nombre
            for nombre, url in _urls_validas(solo).items()
        }
        for f in as_completed(futuros):
            nombre = futuros[f]
            try:
                dfs[nombre] = f.result()
                print(f"cargado: {nombre} {dfs[nombre].shape}")
            except Exception as e:
                fallos[nombre] = e
                print(f"FALLÓ: {nombre} -> {e}")

    if fallos:
        print(f"\nReintenta solo los que fallaron: cargar_todo(solo={list(fallos)})")

    return {n: dfs[n] for n in URLS if n in dfs}
 
dfs = cargar_todo(carpeta=CARPETA_DATOS, dtype=str)
#Dataframes
df_sivigila = dfs.get("sivigila") # Intentos suicidio reportados a SIVIGILA
df_afectados_alertas = dfs.get("afectados_alertas") # Afectados por alertas recibidas en SATMED
df_factores_riesgo = dfs.get("factores_riesgo") # Factores de riesgo identificados en SATMED
df_alertas_factores_riesgo = dfs.get("alertas_factores_riesgo") #
df_defunciones = dfs.get("defunciones") # defunciones /mortalidad





In [ ]:
def cargar_proyecciones(carpeta=None, forzar=False):
    """Población por comuna 2018–2030. Descarga el Excel si no está en la carpeta y lo lee."""
    url = URL_CENCO_PREDICCIONES_PROYECCIONES_POR_COMUNA
    kwargs = {"header": 15, "dtype": {"Código Comuna": str}}   # encabezado en la fila 16

    def descargar():
        datos = _get(url)
        if not datos.startswith(b"PK"):          # un .xlsx es un zip; si no, llegó HTML
            raise ContenidoInvalido("la descarga no es un archivo Excel válido")
        return datos

    if not carpeta:                              # solo memoria
        df = pd.read_excel(io.BytesIO(descargar()), **kwargs)
    else:
        ruta = Path(carpeta) / url.split("/")[-1].split("?")[0]
        if forzar or not ruta.exists():
            ruta.parent.mkdir(parents=True, exist_ok=True)
            parcial = ruta.with_suffix(ruta.suffix + ".part")
            parcial.write_bytes(descargar())
            parcial.replace(ruta)                # renombrado atómico
        df = pd.read_excel(ruta, **kwargs)

    print(f"cargado: proyecciones {df.shape}")
    return df

df_proyecciones = cargar_proyecciones(carpeta=CARPETA_DATOS)

## 2. Carga y limpieza

Decisiones de limpieza, cada una registrada en una bitácora:
- Se eliminan filas duplicadas exactas.
- La comuna se toma del **código** (`codigo_comuna`), porque el nombre trae variantes ("Belen"/"Belén", "La America"/"La América").
- Se excluyen residentes de otro municipio (código 98). Los casos sin comuna (código 99) cuentan para la ciudad pero no para el análisis por comuna.
- Las variables Sí/No vienen codificadas como 1 = Sí, 2 = No; se pasan a 1/0.
- Según el diccionario de datos, `gp_indigen` significa **habitante de calle**; se renombra.

In [ ]:
bitacora = []

def _fuente(df, nombre):
    if df is not None and all(pd.api.types.is_string_dtype(t) for t in df.dtypes):
        return df.copy()
    
    ruta = Path(CARPETA_DATOS) / URLS[nombre].split("/")[-1].split("?")[0]
    if ruta.exists():
        print(f"{nombre}: usando el CSV {ruta}")
        return pd.read_csv(ruta, dtype=str, low_memory=False)

    raise ValueError(f"'{nombre}': no hay variable válida (como texto) ni CSV en {ruta}")

def registrar(paso, valor):
    bitacora.append({"paso": paso, "valor": valor})

def limpiar_intentos_suicidio(df):
    it = _fuente(df, "sivigila")
    #duplicados exactos
    registrar("Intentos: registros originales", len(it))
    n = len(it); it = it.drop_duplicates()
    registrar("Intentos: duplicados exactos eliminados", n - len(it))

    # Comuna
    it["cod_comuna"] = it["codigo_comuna"].str.zfill(2)
    registrar("Intentos: residentes de otro municipio excluidos", int((it.cod_comuna == "98").sum()))
    it = it[it.cod_comuna != "98"].copy()
    it["comuna"] = it["cod_comuna"].map(COMUNAS).fillna("Sin información")
    registrar("Intentos: sin comuna (solo cuentan para la ciudad)", int((it.comuna == "Sin información").sum()))

    # Año, edad, sexo
    it["anio"] = it["año"].astype(int)
    it["edad"] = pd.to_numeric(it["edad_"], errors="coerce")
    it.loc[it["uni_med_"] != "1", "edad"] = 0          # edad en meses o días -> menor de 1 año
    it["sexo"] = it["sexo_"].map({"F": "Mujer", "M": "Hombre"})

    # Variables Sí/No: 1 -> 1, 2 -> 0, otro -> NaN
    SI_NO = ["inten_prev", "idea_suici", "plan_suici", "prob_famil", "prob_parej", "prob_econo",
            "prob_legal", "prob_labor", "prob_consu", "esco_educ", "muerte_fam", "maltr_fps",
            "antec_tran", "abuso_alco", "intoxicaci", "arma_corto", "ahorcamien", "lanz_vacio",
            "lanz_vehic", "arma_fuego", "gp_indigen"]
    for c in SI_NO:
        it[c] = it[c].map({"1": 1, "2": 0})
    # Validar que realmente sea habitante de calle
    it = it.rename(columns={"gp_indigen": "gp_habitante_calle"})

    #Grupos edad
    cortes = [-1, 9, 14, 19, 29, 44, 200]
    etiquetas = ["0–9", "10–14", "15–19", "20–29", "30–44", "45 y más"]
    it["grupo_edad"] = pd.cut(it["edad"], cortes, labels=etiquetas)
    registrar("Intentos: registros finales (residentes de Medellín)", len(it))

    return it

def limpiar_defunciones(df):
    de = _fuente(df, "defunciones")

    # Defunciones: suicidios (CIE-10 X60–X84)
    registrar("Defunciones: registros originales", len(de))
    su = de[de["C_BAS1"].str[:3].between("X60", "X84")].copy()   # CIE-10: lesiones autoinfligidas
    registrar("Defunciones: suicidios (CIE-10 X60–X84)", len(su))
    su["cod_comuna"] = su["COMUNA_RES"].str[:2].where(su["COMUNA_RES"].str[:2].str.isdigit(), "99")
    su["comuna"] = su["cod_comuna"].map(COMUNAS).fillna("Sin información")
    su["anio"] = su["ANO"].astype(int)
    su["sexo"] = su["SEXO"].map({"1": "Hombre", "2": "Mujer"})

    return su

bitacora.clear()
it = limpiar_intentos_suicidio(df_sivigila)
su  = limpiar_defunciones(df_defunciones)
pd.DataFrame(bitacora)

### Calidad de los datos

Dos problemas que afectan la lectura: las variables de factores y mecanismo **solo se diligencian desde 2018**, y en **2022** se dispara el porcentaje de casos sin comuna.

In [ ]:
calidad = pd.DataFrame({
    "casos": it.groupby("anio").size(),
    "% sin comuna": it.groupby("anio")["comuna"].apply(lambda s: (s == "Sin información").mean() * 100).round(1),
    "% sin dato de intento previo": it.groupby("anio")["inten_prev"].apply(lambda s: s.isna().mean() * 100).round(1),
})
calidad

## 3. ¿Cuándo? Tendencia en el tiempo

In [ ]:
anual = pd.DataFrame({"intentos": it.groupby("anio").size(),
                      "suicidios": su.groupby("anio").size()}).reindex(range(2012, 2023))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
a = anual["intentos"].dropna()
ax[0].bar(a.index, a.values, color=[AZUL if y == a.index.max() else GRIS for y in a.index], width=0.65)
ax[0].set_title("Intentos notificados por año")
s = anual["suicidios"].dropna()
ax[1].bar(s.index, s.values, color=[AZUL if y == s.index.max() else GRIS for y in s.index], width=0.65)
ax[1].set_title("Muertes por suicidio por año (CIE-10 X60–X84)")
for axis in ax: axis.set_xticks(range(2012, 2023)); axis.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

c_int = anual.loc[2022, "intentos"] / anual.loc[2014, "intentos"] - 1
c_sui = anual.loc[2021, "suicidios"] / anual.loc[2014, "suicidios"] - 1
print(f"Intentos: {anual.loc[2014,'intentos']:.0f} (2014) -> {anual.loc[2022,'intentos']:.0f} (2022), cambio {c_int:+.0%}")
print(f"Suicidios: {anual.loc[2014,'suicidios']:.0f} (2014) -> {anual.loc[2021,'suicidios']:.0f} (2021), cambio {c_sui:+.0%}")

In [ ]:
edad = pd.crosstab(it.anio, it.grupo_edad)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for g, col in zip(["10–14", "15–19", "20–29", "30–44", "45 y más"],
                  ["#e08a3c", "#b8463b", AZUL, "#5b9a8b", GRIS]):
    ax[0].plot(edad.index, edad[g], marker="o", ms=3, lw=2, color=col, label=g)
ax[0].set_title("Intentos por grupo de edad"); ax[0].legend(frameon=False, fontsize=8)
sexo = pd.crosstab(it.anio, it.sexo)
ax[1].plot(sexo.index, sexo["Mujer"], marker="o", ms=3, lw=2, color=AZUL, label="Mujeres")
ax[1].plot(sexo.index, sexo["Hombre"], marker="o", ms=3, lw=2, color="#e08a3c", label="Hombres")
ax[1].set_title("Intentos por sexo"); ax[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

jov = it.grupo_edad.isin(["10–14", "15–19", "20–29"]).mean()
print(f"Personas de 10 a 29 años: {jov:.0%} de los intentos")
print(f"Mujeres: {(it.sexo=='Mujer').mean():.0%} de los intentos | Hombres: {(su.sexo=='Hombre').mean():.0%} de los suicidios")

## 4. ¿Dónde? Diferencias entre comunas

Se comparan dos periodos de cuatro años (2014–2017 y 2018–2021) para suavizar la variación de un año a otro. 2022 queda fuera de esta comparación por su alto porcentaje de casos sin comuna.

El indicador **intentos por cada suicidio** no depende del tamaño de la población: un valor bajo puede señalar menos notificación de intentos.

In [ ]:
ca = pd.DataFrame({"intentos": it.groupby(["comuna", "anio"]).size(),
                   "suicidios": su.groupby(["comuna", "anio"]).size()}).fillna(0).reset_index()
ca = ca[ca.comuna != "Sin información"]

def suma(df, col, a, b):
    return df[df.anio.between(a, b)].groupby("comuna")[col].sum()

com = pd.DataFrame({
    "int_2014_2017": suma(ca, "intentos", 2014, 2017),
    "int_2018_2021": suma(ca, "intentos", 2018, 2021),
    "sui_2014_2021": suma(ca, "suicidios", 2014, 2021),
})
com["cambio_%"] = ((com.int_2018_2021 / com.int_2014_2017 - 1) * 100).round(0)
com["intentos_por_suicidio"] = ((com.int_2014_2017 + com.int_2018_2021) / com.sui_2014_2021).round(1)
com = com.sort_values("cambio_%", ascending=False)
ciudad = com.int_2018_2021.sum() / com.int_2014_2017.sum() - 1
print(f"Cambio en toda la ciudad entre periodos: {ciudad:+.0%}")
com

In [ ]:
d = com.sort_values("cambio_%")
fig, ax = plt.subplots(figsize=(8, 7))
y = np.arange(len(d))
ax.hlines(y, d.int_2014_2017, d.int_2018_2021, color=GRIS, lw=2)
ax.scatter(d.int_2014_2017, y, color=GRIS, s=40, label="2014–2017", zorder=3)
ax.scatter(d.int_2018_2021, y, color=[ROJO if c > ciudad * 100 else AZUL for c in d["cambio_%"]], s=40,
           label="2018–2021", zorder=3)
for yi, (c, x) in enumerate(zip(d["cambio_%"], d[["int_2014_2017", "int_2018_2021"]].max(axis=1))):
    ax.text(x + 12, yi, f"{c:+.0f} %", va="center", fontsize=8)
ax.set_yticks(y); ax.set_yticklabels(d.index)
ax.set_title("Intentos por comuna: 2014–2017 frente a 2018–2021\n(rojo: crece más que el promedio de la ciudad)")
ax.legend(frameon=False, loc="lower right"); ax.grid(axis="y", alpha=0)
plt.tight_layout(); plt.show()

## 5. ¿Qué hay detrás? Factores reportados, 2018–2022

Porcentaje sobre los casos que tienen el dato. Un caso puede tener varios factores.

In [ ]:
r = it[it.anio >= 2018]
grupos = {
    "Desencadenante": {"Problemas de pareja": "prob_parej", "Problemas familiares*": "prob_famil",
                       "Consumo de sustancias": "prob_consu", "Problemas económicos": "prob_econo",
                       "Problemas escolares": "esco_educ", "Maltrato": "maltr_fps"},
    "Factor de riesgo": {"Trastorno psiquiátrico previo": "antec_tran", "Intento previo": "inten_prev",
                         "Ideación suicida": "idea_suici", "Plan suicida": "plan_suici",
                         "Abuso de alcohol": "abuso_alco"},
    "Mecanismo": {"Intoxicación": "intoxicaci", "Arma cortopunzante": "arma_corto",
                  "Ahorcamiento": "ahorcamien", "Lanzamiento al vacío": "lanz_vacio"},
}
fig, axs = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, (titulo, vars_) in zip(axs, grupos.items()):
    v = pd.Series({k: r[c].mean() * 100 for k, c in vars_.items()}).sort_values()
    ax.barh(v.index, v.values, color=AZUL, height=0.6)
    for i, val in enumerate(v.values): ax.text(val + 1, i, f"{val:.0f} %", va="center", fontsize=8)
    ax.set_title(titulo); ax.set_xlim(0, 100); ax.grid(axis="y", alpha=0)
plt.tight_layout(); plt.show()
print("* Problemas familiares: diligenciado solo en", f"{r.prob_famil.notna().mean():.0%}", "de los casos.")
print(f"Casos con intento previo (2018–2022): {r.inten_prev.mean():.0%}")

## 6. Tasas por 100.000 habitantes

Para comparar comunas de distinto tamaño se necesita la población. Usamos el Excel de **proyecciones por comuna 2018–2030** del Departamento Administrativo de Planeación (DANE, actualización 2025). Los corregimientos vienen partidos en cabecera y zona rural: se suman. Como la población empieza en 2018, las tasas cubren 2018–2022 para intentos y 2018–2021 para suicidios.

**Tasa = casos ÷ población × 100.000.**

In [ ]:
x = df_proyecciones.copy()
x = x[x["Código Comuna"].isin(COMUNAS.keys())]
anios = [c for c in x.columns if str(c).split(".")[0].isdigit()]
pob = (x.groupby("Código Comuna")[anios].sum()          # suma cabecera + rural de cada corregimiento
         .stack().rename("poblacion").reset_index()
         .rename(columns={"Código Comuna": "cod_comuna", "level_1": "anio"}))
pob["anio"] = pob["anio"].astype(float).astype(int)
pob["comuna"] = pob["cod_comuna"].map(COMUNAS)

# Tasa de la ciudad (incluye casos sin comuna)
pob_ciudad = pob.groupby("anio").poblacion.sum()
tasa_ciudad = pd.DataFrame({
    "intentos": anual.intentos, "suicidios": anual.suicidios, "poblacion": pob_ciudad}).dropna(subset=["poblacion"])
tasa_ciudad["tasa_intentos"] = (tasa_ciudad.intentos / tasa_ciudad.poblacion * 1e5).round(1)
tasa_ciudad["tasa_suicidios"] = (tasa_ciudad.suicidios / tasa_ciudad.poblacion * 1e5).round(1)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ti = tasa_ciudad.tasa_intentos.dropna(); ts = tasa_ciudad.tasa_suicidios.dropna()
ax[0].plot(ti.index, ti.values, marker="o", lw=2, color=AZUL); ax[0].set_ylim(0, ti.max() * 1.2)
ax[0].set_title("Tasa de intentos (por 100.000 hab.)")
ax[1].plot(ts.index, ts.values, marker="o", lw=2, color=ROJO); ax[1].set_ylim(0, ts.max() * 1.3)
ax[1].set_title("Tasa de suicidio (por 100.000 hab.)")
for axis, s_ in [(ax[0], ti), (ax[1], ts)]:
    axis.set_xticks(s_.index)
    for xx, yy in s_.items(): axis.annotate(f"{yy:.1f}", (xx, yy), xytext=(0, 6), textcoords="offset points", ha="center", fontsize=8)
plt.tight_layout(); plt.show()
tasa_ciudad[["intentos", "suicidios", "poblacion", "tasa_intentos", "tasa_suicidios"]]

In [ ]:
# Tasas por comuna, periodo 2018–2021 (2022 se excluye por los casos sin comuna)
t = ca.merge(pob[["comuna", "anio", "poblacion"]], on=["comuna", "anio"])
p = t[t.anio.between(2018, 2021)].groupby("comuna")[["intentos", "suicidios", "poblacion"]].sum()
p["tasa_intentos"] = (p.intentos / p.poblacion * 1e5).round(1)
p["tasa_suicidios"] = (p.suicidios / p.poblacion * 1e5).round(1)
mediana = p.tasa_intentos.median()

d = p.sort_values("tasa_intentos")
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.barh(d.index, d.tasa_intentos, color=[AZUL if v > mediana else GRIS for v in d.tasa_intentos], height=0.65)
ax.axvline(mediana, color="#666", ls="--", lw=1); ax.text(mediana + 1, -0.9, f"mediana {mediana:.0f}", fontsize=8)
for i, v in enumerate(d.tasa_intentos): ax.text(v + 1, i, f"{v:.0f}", va="center", fontsize=8)
ax.set_title("Tasa de intentos por comuna, 2018–2021 (por 100.000 hab. al año)"); ax.grid(axis="y", alpha=0)
plt.tight_layout(); plt.show()
print("Ojo: Palmitas, Santa Elena y Altavista tienen pocos casos; su tasa cambia mucho con pocos eventos.")

## 7. Priorización exploratoria

Cada comuna se ubica según dos indicadores:
- **Tasa alta:** tasa de intentos 2018–2021 por encima de la mediana de las comunas.
- **Tendencia creciente:** los intentos de 2018–2021 crecieron más que el promedio de la ciudad frente a 2014–2017. Se usan ocho años porque la tendencia de solo 2018–2021 queda distorsionada por la caída de 2020 (pandemia). La población de cada comuna creció poco en esos años, así que el cambio en conteos se parece al cambio en la tasa.

**Alta** = cumple ambos · **Media** = cumple uno · **Seguimiento** = ninguno · **Pocos casos** = menos de 100 intentos en 2018–2021 (tasa inestable).

In [ ]:
pri = p[["tasa_intentos", "tasa_suicidios", "intentos"]].join(com[["cambio_%"]])
alta, crece = pri.tasa_intentos > mediana, pri["cambio_%"] > ciudad * 100
pri["prioridad"] = np.select([pri.intentos < 100, alta & crece, alta | crece], ["Pocos casos", "Alta", "Media"], "Seguimiento")
orden = {"Alta": 0, "Media": 1, "Seguimiento": 2, "Pocos casos": 3}
pri = pri.sort_values(["prioridad", "tasa_intentos"], key=lambda s: s.map(orden) if s.name == "prioridad" else -s)

col = {"Alta": ROJO, "Media": "#e08a3c", "Seguimiento": AZUL, "Pocos casos": GRIS}
fig, ax = plt.subplots(figsize=(9, 6))
for nivel, g in pri.groupby("prioridad"):
    ax.scatter(g.tasa_intentos, g["cambio_%"], s=60, color=col[nivel], label=nivel, zorder=3, edgecolor="white")
for nombre, f in pri.iterrows():
    ax.annotate(nombre, (f.tasa_intentos, f["cambio_%"]), fontsize=7.5, xytext=(5, 3), textcoords="offset points")
ax.axvline(mediana, color="#666", ls="--", lw=1); ax.axhline(ciudad * 100, color="#666", ls="--", lw=1)
ax.text(ax.get_xlim()[1], ciudad * 100 + 1.5, "promedio ciudad", ha="right", fontsize=8, color="#666")
ax.set_xlabel("Tasa de intentos 2018–2021 (por 100.000 hab. al año)")
ax.set_ylabel("Cambio en intentos: 2018–2021 vs 2014–2017 (%)")
ax.set_title("Matriz de priorización exploratoria")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.show()
print("Prioridad alta:", ", ".join(pri.index[pri.prioridad == "Alta"]))
pri

## 8. Contexto: alertas tempranas SALVAVIDAS

[SALVAVIDAS](https://medata.app.medellin.gov.co/node/16832) es el Sistema de Alertas Tempranas de Medellín para niños, niñas, adolescentes y jóvenes (MEData, Secretaría de la Juventud). No registra intentos de suicidio, pero sí alertas con factores de **salud mental**. La pregunta es si las comunas con más intentos también concentran más alertas de este tipo.

Cómo se usa:
- Se unen **afectados** y **alertas asociadas a factores** por `ALERTA_ID`, y el **catálogo de factores** por `FACTOR_RIESGO_ID`.
- No se usa la tabla de alertas recibidas: no trae el identificador de la alerta y contiene datos de contacto y nombres de funcionarios que no se necesitan.
- Factores de salud mental: estado de ánimo y salud mental (1), comportamiento suicida (34), autolesión (35) y trastorno de conducta o salud mental (36). Los factores 34 a 36 solo aparecen desde 2023.
- Periodo 2018–2022, el mismo de las tasas de intentos. La comparación es **ecológica** (comuna contra comuna), nunca persona a persona.

In [ ]:
afe = _fuente(df_afectados_alertas, "afectados_alertas")        
aso = _fuente(df_alertas_factores_riesgo, "alertas_factores_riesgo")   
cat = _fuente(df_factores_riesgo, "factores_riesgo")    

SALUD_MENTAL = ["1", "34", "35", "36"]
display(cat[cat.id.isin(SALUD_MENTAL)][["id", "DESCRIPCION"]])

afe["cod_comuna"] = afe["COMUNA_NAME"].str.extract(r"Comuna (\d+)")[0].str.zfill(2)
comuna_alerta = afe.dropna(subset=["cod_comuna"]).drop_duplicates("ALERTA_ID")[["ALERTA_ID", "cod_comuna"]]
aso["anio"] = aso["createdAt"].str[:4].astype(int)
sm = (aso[aso.FACTOR_RIESGO_ID.isin(SALUD_MENTAL) & aso.anio.between(2018, 2022)]
        .drop_duplicates("ALERTA_ID").merge(comuna_alerta, on="ALERTA_ID"))
print(f"Alertas con comuna registrada: {comuna_alerta.ALERTA_ID.nunique()} de {afe.ALERTA_ID.nunique()} "
      f"({comuna_alerta.ALERTA_ID.nunique() / afe.ALERTA_ID.nunique():.0%})")
print(f"Alertas de salud mental 2018–2022 con comuna: {len(sm)}")

pob_18_22 = pob[pob.anio.between(2018, 2022)].groupby("cod_comuna").poblacion.sum()
sv = sm.groupby("cod_comuna").size().rename("alertas_salud_mental").to_frame().join(pob_18_22)
sv["tasa_alertas"] = (sv.alertas_salud_mental / sv.poblacion * 1e5).round(1)
sv.index = sv.index.map(COMUNAS)
cmp_ = pri[["tasa_intentos", "prioridad"]].join(sv[["alertas_salud_mental", "tasa_alertas"]])

# Asociación de rangos (Spearman) sin las comunas con pocos casos, con valor p por permutación
base = cmp_[cmp_.prioridad != "Pocos casos"].dropna()
rho = base.tasa_intentos.rank().corr(base.tasa_alertas.rank())
rng = np.random.default_rng(2026)
perm = [base.tasa_intentos.rank().corr(pd.Series(rng.permutation(base.tasa_alertas.rank().values), index=base.index))
        for _ in range(10000)]
p_valor = (np.sum(np.abs(perm) >= abs(rho)) + 1) / (len(perm) + 1)
print(f"Spearman entre tasa de intentos y tasa de alertas de salud mental: {rho:.2f} (n = {len(base)}, p = {p_valor:.4f})")

fig, ax = plt.subplots(figsize=(8, 5.5))
col = base.prioridad.map({"Alta": ROJO, "Media": "#e08a3c", "Seguimiento": AZUL})
ax.scatter(base.tasa_intentos, base.tasa_alertas, c=col, s=55, zorder=3, edgecolor="white")
for nombre, f in base.iterrows():
    off, ha = {"San Antonio de Prado": ((5, -10), "left"), "San Cristóbal": ((-6, 3), "right")}.get(nombre, ((5, 3), "left"))
    ax.annotate(nombre, (f.tasa_intentos, f.tasa_alertas), fontsize=7.5, xytext=off, textcoords="offset points", ha=ha)
for nivel, c in [("Alta", ROJO), ("Media", "#e08a3c"), ("Seguimiento", AZUL)]:
    ax.scatter([], [], color=c, s=40, label=f"Prioridad {nivel.lower()}")
ax.legend(frameon=False, loc="lower right", fontsize=8)
ax.set_xlabel("Tasa de intentos 2018–2021 (por 100.000 hab. al año)")
ax.set_ylabel("Alertas SALVAVIDAS de salud mental\n2018–2022 (por 100.000 hab.-año)")
ax.set_title(f"Comunas con más intentos también registran más alertas de salud mental (Spearman {rho:.2f})")
plt.tight_layout(); plt.show()
cmp_.sort_values("tasa_alertas", ascending=False)

**Lectura.** La asociación es **exploratoria y ecológica**: muestra que, entre comunas, a más intentos notificados también hay más alertas de salud mental en SALVAVIDAS. No dice que las mismas personas estén en las dos bases ni que una cosa cause la otra. Sí muestra algo útil para la recomendación: en las comunas priorizadas ya existe un sistema de alertas activo que puede apoyar el seguimiento.

## 9. Proyección de intentos a 2026

Ajustamos una regresión lineal simple con el año como variable independiente ($X$) y el número de intentos notificados como variable dependiente ($y$), con los datos de 2014 a 2022, para proyectar la tendencia hasta 2026.

Junto a cada valor proyectado mostramos un **rango probable (intervalo de predicción del 95 %)**. Es amplio porque la serie tiene solo 9 años y varía mucho de un año a otro (por ejemplo, la caída de 2020 y el salto de 2022). Por eso la proyección para 2023 queda por debajo del valor real de 2022: la línea recta sigue la tendencia promedio y no el último salto. Se proyectan casos y no tasas porque la población por comuna solo está disponible desde 2018.

In [ ]:
from sklearn.linear_model import LinearRegression
from scipy import stats

# Serie anual de la sección 3 (`anual`), con los datos ya limpios, 2014-2022.
# Se proyectan casos y no tasas: la población por comuna solo existe desde 2018 (5 años, muy pocos para una tendencia).
datos_modelo = anual.loc[2014:2022, 'intentos'].dropna().reset_index()
datos_modelo.columns = ['anio', 'intentos']
X = datos_modelo[['anio']]
y = datos_modelo['intentos']

modelo = LinearRegression().fit(X, y)

# Proyección 2023-2026
anios_futuros = pd.DataFrame({'anio': [2023, 2024, 2025, 2026]})
predicciones = modelo.predict(anios_futuros)

# Intervalo de predicción del 95 %: rango donde probablemente caerá el valor real de cada año
n = len(datos_modelo)
residuos = y - modelo.predict(X)
error_tipico = np.sqrt((residuos ** 2).sum() / (n - 2))
x = datos_modelo['anio']
sxx = ((x - x.mean()) ** 2).sum()
t = stats.t.ppf(0.975, n - 2)
margen = t * error_tipico * np.sqrt(1 + 1 / n + (anios_futuros['anio'] - x.mean()) ** 2 / sxx)

proyecciones = pd.DataFrame({
    'anio': anios_futuros['anio'],
    'intentos_estimados': predicciones.round(0).astype(int),
    'minimo_95': (predicciones - margen).round(0).astype(int),
    'maximo_95': (predicciones + margen).round(0).astype(int),
})
display(proyecciones)


Grafiquemos la serie histórica junto con la línea de tendencia estimada y las proyecciones futuras para visualizar el comportamiento esperado.

In [ ]:
import matplotlib.pyplot as plt

# Asegurar variables de diseño locales si es necesario
AZUL, GRIS, ROJO = "#2f5d8a", "#b9c0c8", "#b8463b"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})

fig, ax = plt.subplots(figsize=(8, 4.5))

# Datos históricos
ax.plot(datos_modelo['anio'], datos_modelo['intentos'], marker='o', color=AZUL, label='Histórico (SIVIGILA)')

# Línea de tendencia ajustada (histórica)
y_ajustado = modelo.predict(X)
ax.plot(datos_modelo['anio'], y_ajustado, color=GRIS, linestyle='--', label='Tendencia ajustada')

# Intervalo de predicción del 95 %
ax.fill_between(proyecciones['anio'], proyecciones['minimo_95'], proyecciones['maximo_95'],
                color=ROJO, alpha=0.12, label='Rango probable (95 %)')

# Proyecciones futuras
ax.plot(proyecciones['anio'], proyecciones['intentos_estimados'], marker='s', color=ROJO, linestyle=':', label='Proyección (2023-2026)')

# Personalización
ax.set_title('Tendencia y Proyección de Intentos de Suicidio en Medellín')
ax.set_xlabel('Año')
ax.set_ylabel('Número de Intentos')
ax.set_xticks(range(2014, 2027))
ax.legend(frameon=False)

# Anotaciones en los puntos proyectados
for index, row in proyecciones.iterrows():
    ax.annotate(f"{row['intentos_estimados']}", (row['anio'], row['intentos_estimados']), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=8, color=ROJO)

plt.tight_layout()
plt.show()

# Mostrar métricas del modelo
ultimo = int(datos_modelo['intentos'].iloc[-1])
print(f"Último año observado (2022): {ultimo} intentos")
r2 = modelo.score(X, y)
print(f"Coeficiente de determinación (R²): {r2:.2f}")
print(f"Crecimiento anual estimado por el modelo: {modelo.coef_[0]:.1f} intentos por año")
print(f"Proyección 2026: {proyecciones.intentos_estimados.iloc[-1]} intentos "
      f"(rango probable {proyecciones.minimo_95.iloc[-1]}–{proyecciones.maximo_95.iloc[-1]})")

## 10. Factores por comuna, 2018–2022

Calcularemos la tasa de prevalencia de los principales factores reportados en cada comuna para identificar si existen patrones geográficos o problemáticas específicas que requieran enfoques diferenciados en la prevención.

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np

# COMUNAS, it['comuna'] y los factores en 0/1 ya se prepararon en la sección 2

# Definir factores clave y asegurar su conversión a numérico (1 = Sí, 0 = No)
factores_clave = {
    'Pareja': 'prob_parej',
    'Familiar': 'prob_famil',
    'Económico': 'prob_econo',
    'Consumo Sustancias': 'prob_consu',
    'Escolar/Educativo': 'esco_educ',
    'Trastorno Psiquiátrico': 'antec_tran',
    'Intento Previo': 'inten_prev'
}

# Filtrar datos a partir de 2018 y excluir casos sin información de comuna
r_comunas = it[(it['anio'] >= 2018) & (it['comuna'] != 'Sin información')].copy()

# Calcular el porcentaje de presencia de cada factor por comuna
comuna_factores = {}
for etiqueta, columna in factores_clave.items():
    # Convertir a flotante para calcular la media
    comuna_factores[etiqueta] = r_comunas.groupby('comuna')[columna].mean() * 100

# Construir el DataFrame final
df_factores_comuna = pd.DataFrame(comuna_factores).round(1)

# Comunas con pocos casos (sección 7): sus porcentajes cambian mucho con pocos eventos.
# Se marcan con * en tablas y gráficos.
POCOS_CASOS = set(pri.index[pri.prioridad == "Pocos casos"])

def marcar(nombre):
    return f"{nombre} *" if nombre in POCOS_CASOS else nombre

# Tabla ordenada por problemas de pareja, con el número de casos de cada comuna
casos_comuna = r_comunas.groupby('comuna').size().rename('Casos 2018–2022')
tabla_factores = df_factores_comuna.join(casos_comuna).sort_values(by='Pareja', ascending=False)
tabla_factores.index = tabla_factores.index.map(marcar)
display(tabla_factores)
print(f"* Pocos casos ({', '.join(sorted(POCOS_CASOS))}): sus porcentajes pueden variar mucho; leerlos con cautela.")

Visualizaremos esta distribución mediante un mapa de calor para identificar de manera inmediata qué comunas concentran problemáticas específicas de salud mental, adicciones o vulnerabilidad socioeconómica.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Asegurar las variables de diseño estéticas globales
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})

datos_heatmap = df_factores_comuna.sort_index()
datos_heatmap.index = datos_heatmap.index.map(marcar)   # * = pocos casos

plt.figure(figsize=(10, 8))
sns.heatmap(
    datos_heatmap,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    cbar_kws={'label': '% de casos con el factor presente'},
    linewidths=0.5
)

plt.title('Prevalencia de Factores de Riesgo y Desencadenantes por Comuna (2018-2022)')
plt.xlabel('Factores Evaluados')
plt.ylabel('Comuna  (* = pocos casos)')
plt.tight_layout()
plt.show()

## 11. ¿Qué factores se asocian con la tasa de intentos?

Para identificar qué factores de riesgo se asocian más con las tasas de intentos en el territorio, unimos la prevalencia de factores (sección 10) con las tasas 2018–2021 de la sección 6, sin las comunas con pocos casos. Usamos la **correlación de Spearman**, igual que en la sección 8, porque con solo 18 comunas es más robusta que Pearson frente a valores extremos. Como se prueban 7 factores a la vez, además del valor p mostramos el **valor p ajustado por Holm**: si solo el valor p sin ajustar es menor que 0,05, la asociación debe tomarse como una señal a revisar, no como un hallazgo confirmado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Tasas de intentos 2018-2021 por comuna calculadas en la sección 6 con la población
#    real del Excel de proyecciones (DAP/DANE). Se excluyen las comunas marcadas como
#    "Pocos casos" en la sección 7, cuya tasa es inestable.
tasas_estables = pri.loc[pri.prioridad != "Pocos casos", "tasa_intentos"]

# 2. Unir tasas con prevalencia de factores
df_analisis = df_factores_comuna.join(tasas_estables, how='inner')

# 3. Correlación de Spearman (por rangos, más robusta con solo 18 comunas), como en la sección 8.
#    Valor p por permutación y corrección de Holm, porque se prueban 7 factores a la vez.
rng = np.random.default_rng(2026)
rango_tasa = df_analisis['tasa_intentos'].rank().values
permutaciones = np.array([rng.permutation(rango_tasa) for _ in range(10000)])

filas = []
for factor in df_factores_comuna.columns:
    rango_factor = df_analisis[factor].rank().values
    rho = np.corrcoef(rango_factor, rango_tasa)[0, 1]
    rho_perm = np.array([np.corrcoef(rango_factor, p)[0, 1] for p in permutaciones])
    p_valor = (np.sum(np.abs(rho_perm) >= abs(rho)) + 1) / (len(rho_perm) + 1)
    filas.append({'factor': factor, 'rho': rho, 'p': p_valor})

resultado = pd.DataFrame(filas).set_index('factor').sort_values('rho', ascending=False)

# Corrección de Holm: ajusta los valores p por haber hecho varias pruebas
orden = resultado['p'].sort_values()
m = len(orden)
resultado['p_holm'] = (orden * np.arange(m, 0, -1)).cummax().clip(upper=1)

correlaciones = resultado['rho']
print(f"Comunas incluidas: {len(df_analisis)}")
display(resultado.round(3).rename(columns={'rho': 'Spearman (rho)', 'p': 'Valor p',
                                           'p_holm': 'Valor p ajustado (Holm)'}))


Grafiquemos estas correlaciones para facilitar la interpretación de cuáles son los factores con mayor peso o asociación estadística sobre las tasas de intentos en Medellín.

In [ ]:
# Definir colores consistentes con el diseño del notebook
AZUL, ROJO = "#2f5d8a", "#b8463b"
# Rojo: valor p < 0,05 sin ajustar (señal a revisar); azul: sin asociación clara
colores = [ROJO if p < 0.05 else AZUL for p in resultado['p']]

fig, ax = plt.subplots(figsize=(8, 4.5))
correlaciones.plot(kind='barh', color=colores, ax=ax, width=0.6)

ax.set_title('Asociación (Correlación) entre Factores de Riesgo y la Tasa de Intentos por Comuna')
ax.set_xlabel('Correlación de Spearman (rho)')
ax.set_ylabel('Factor Evaluado')
ax.set_xlim(-1, 1)
ax.axvline(0, color='#666', linestyle='-', linewidth=0.8)

# Anotaciones con los valores exactos
for i, val in enumerate(correlaciones.values):
    ax.text(val + 0.02 if val >= 0 else val - 0.08, i, f"{val:+.2f}", va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 12. Mapas por comuna

Para ver la prevalencia de los factores de riesgo sobre el territorio, pintamos las comunas de Medellín con `folium`, usando los límites oficiales (GeoJSON) de la Alcaldía. Primero se muestra el factor **económico**, el que tiene la asociación más alta con la tasa de intentos en la sección 11, aunque es débil y no significativa. Al pasar el mouse sobre una comuna se ve su nombre y porcentaje. Si el GeoJSON no se puede descargar, se muestra un mapa de burbujas de respaldo. El mapa también se guarda como `mapa_factor_economico.html`.

In [ ]:
%pip install folium

In [ ]:
# Preparación: límites de las comunas y funciones para unirlos con los datos
import folium
import json
import urllib.request
import pandas as pd
from branca.colormap import linear

# 1. GeoJSON oficial de la Alcaldía de Medellín (capa "Comuna Catastral", 16 comunas + 5 corregimientos)
#    outSR=4326 devuelve las coordenadas en latitud/longitud, que es lo que usa folium
URL_GEOJSON = ("https://www.medellin.gov.co/servidormapas/rest/services/ccio_ind_turism/"
               "VA_Establecimiento_Comercial/MapServer/5/query"
               "?where=1%3D1&outFields=comuna,nombre&outSR=4326&f=geojson")
RUTA_GEOJSON = "comunas_medellin.geojson"

# Diccionario de códigos oficiales y nombres para mapeo y consistencia
NOMBRES = {
    1: "Popular", 2: "Santa Cruz", 3: "Manrique", 4: "Aranjuez", 5: "Castilla",
    6: "Doce de Octubre", 7: "Robledo", 8: "Villa Hermosa", 9: "Buenos Aires",
    10: "La Candelaria", 11: "Laureles", 12: "La América", 13: "San Javier",
    14: "El Poblado", 15: "Guayabal", 16: "Belén",
    50: "Palmitas", 60: "San Cristóbal", 70: "Altavista", 80: "San Antonio de Prado",
    90: "Santa Elena"
}

# Invertir el diccionario para buscar códigos por nombre
NOMBRES_A_CODIGO = {v.lower(): k for k, v in NOMBRES.items()}

def codigo_comuna(identificador):
    """Normaliza y extrae el código numérico de una comuna."""
    if identificador is None:
        return None
    s = str(identificador).strip().lower()
    # Si es numérico directo
    if s.isdigit():
        val = int(s)
        return val if val in NOMBRES else None
    # Si es el nombre de la comuna
    if s in NOMBRES_A_CODIGO:
        return NOMBRES_A_CODIGO[s]
    return None

def preparar_datos(df):
    """Prepara y limpia el DataFrame para el mapeo geográfico."""
    df = df.copy()
    if "comuna" not in df.columns:
        df = df.reset_index().rename(columns={"index": "comuna"})

    # Crear comuna_id mapeando el nombre o código
    df["comuna_id"] = df["comuna"].apply(codigo_comuna)
    df = df.dropna(subset=["comuna_id"])
    df["comuna_id"] = df["comuna_id"].astype(int)

    # Asegurar que no tengamos duplicados por comuna_id
    duplicadas = df[df.duplicated(subset=["comuna_id"])]['comuna'].tolist()
    if duplicadas:
        print(f"Advertencia: Hay varias filas para estas comunas: {sorted(duplicadas)}.")
        df = df.groupby("comuna_id").first().reset_index()

    return df


In [ ]:
# Dibujar el mapa (usa URL_GEOJSON / RUTA_GEOJSON y las funciones de la celda anterior)
import os
def leer_geojson():
    try:
        if not os.path.exists(RUTA_GEOJSON):
            urllib.request.urlretrieve(URL_GEOJSON, RUTA_GEOJSON)
        with open(RUTA_GEOJSON, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"No se pudo obtener el GeoJSON ({e}). Usaremos el mapa de burbujas de respaldo.")
        return None

def crear_mapa(df_origen, factor_elegido="Económico", leyenda=None, unidad=" %"):
    """Pinta las comunas según la columna `factor_elegido`. `leyenda` y `unidad` permiten
    usarlo también con valores que no son porcentajes (por ejemplo, tasas)."""
    leyenda = leyenda or f"Casos con factor: {factor_elegido} (%)"
    df = preparar_datos(df_origen)
    geo = leer_geojson()

    # Ubicaciones de referencia aproximadas (Latitud, Longitud) de las comunas para fallback
    coordenadas_comunas = {
        1: [6.2878, -75.5456], 2: [6.2995, -75.5512], 3: [6.2755, -75.5461], 4: [6.2789, -75.5583],
        5: [6.2891, -75.5684], 6: [6.3012, -75.5723], 7: [6.2753, -75.5921], 8: [6.2492, -75.5423],
        9: [6.2369, -75.5492], 10: [6.2483, -75.5661], 11: [6.2442, -75.5891], 12: [6.2515, -75.6012],
        13: [6.2523, -75.6134], 14: [6.2089, -75.5654], 15: [6.2163, -75.5791], 16: [6.2234, -75.5991],
        50: [6.3115, -75.6812], 60: [6.2714, -75.6291], 70: [6.2189, -75.6212], 80: [6.1754, -75.6191],
        90: [6.2554, -75.5012]
    }

    mapa = folium.Map(location=[6.2518, -75.5636], zoom_start=12,
                      tiles="Esri.WorldGrayCanvas", control_scale=True)

    if geo:
        valores = df.set_index("comuna_id")[factor_elegido].to_dict()
        capas = []

        for feature in geo["features"]:
            propiedades = feature.get("properties") or {}
            codigo = None
            for campo in ("Numero_Comuna", "numero_comuna", "NUMERO", "codigo", "CODIGO", "cod_comuna", "COMUNA", "SUBAN_CODI"):
                if campo in propiedades:
                    codigo = codigo_comuna(propiedades[campo])
                    if codigo is not None:
                        break
            if codigo is None:
                for campo in ("Nombre_Comuna", "NOMBRE", "nombre", "comuna", "nombre_comuna", "SUBAN_NOMB"):
                    if campo in propiedades:
                        codigo = codigo_comuna(propiedades[campo])
                        if codigo is not None:
                            break

            if codigo not in NOMBRES or not feature.get("geometry"):
                continue

            valor = valores.get(codigo)
            valido = pd.notna(valor)
            feature["properties"] = {
                "comuna_id": codigo,
                "comuna_nombre": f"{codigo:02d} - {NOMBRES[codigo]}",
                "valor_pct": float(valor) if valido else None,
                "valor_texto": f"{valor:.1f}{unidad}" if valido else "Sin datos",
            }
            capas.append(feature)

        geo_mapa = {"type": "FeatureCollection", "features": capas}
        valores_validos = [v for v in valores.values() if pd.notna(v)]
        min_val = min(valores_validos) if valores_validos else 0
        max_val = max(valores_validos) if valores_validos else 100

        escala_color = linear.YlOrRd_09.scale(min_val, max_val)
        escala_color.caption = leyenda

        def estilo(feature):
            valor = feature["properties"]["valor_pct"]
            return {
                "fillColor": escala_color(valor) if valor is not None else "#bdbdbd",
                "color": "#4b5563",
                "weight": 1.2,
                "fillOpacity": 0.78 if valor is not None else 0.4,
            }

        capa = folium.GeoJson(
            geo_mapa,
            name=f"Factor {factor_elegido} (%)",
            style_function=estilo,
            highlight_function=lambda _feature: {"weight": 3, "color": "#00304D", "fillOpacity": 0.9},
            tooltip=folium.GeoJsonTooltip(
                fields=["comuna_nombre", "valor_texto"],
                aliases=["Comuna:", f"{factor_elegido}:"],
                labels=True,
                sticky=True,
            ),
        ).add_to(mapa)
        mapa.fit_bounds(capa.get_bounds())
        escala_color.add_to(mapa)
    else:
        # Fallback de marcadores interactivos proporcionales si el GeoJSON externo falla de forma absoluta
        print("Cargando visualización de contingencia interactiva con burbujas geográficas de datos:")
        valores = df.set_index("comuna_id")[factor_elegido].to_dict()
        valores_validos = [v for v in valores.values() if pd.notna(v)]
        min_val = min(valores_validos) if valores_validos else 1
        max_val = max(valores_validos) if valores_validos else 100

        escala_color = linear.YlOrRd_09.scale(min_val, max_val)
        escala_color.caption = leyenda

        for cid, val in valores.items():
            if cid in coordenadas_comunas and pd.notna(val):
                coords = coordenadas_comunas[cid]
                folium.CircleMarker(
                    location=coords,
                    radius=8 + (val / max_val * 15), # Tamaño relativo al porcentaje
                    popup=f"<b>{NOMBRES[cid]}</b><br>Prevalencia de {factor_elegido}: {val:.1f}%",
                    color="#2f5d8a",
                    fill=True,
                    fill_color=escala_color(val),
                    fill_opacity=0.85
                ).add_to(mapa)
        escala_color.add_to(mapa)

    return mapa

# Crear y mostrar el mapa con el Factor Económico (mayor correlación)
mapa_final = crear_mapa(df_factores_comuna, "Económico")
if mapa_final:
    mapa_final.save("mapa_factor_economico.html")  # se puede abrir en el navegador
    display(mapa_final)

Repetimos el mapa con el **consumo de sustancias**, el segundo factor más asociado con la tasa de intentos (Spearman 0,36, también débil). Si lo comparas con el mapa anterior, puedes ver si las comunas con más casos ligados a problemas económicos son las mismas donde pesa más el consumo. El mapa se guarda como `mapa_factor_consumo.html`.

*Palmitas, Santa Elena y Altavista tienen pocos casos: sus porcentajes pueden variar mucho.*

In [ ]:
# Segundo mapa: consumo de sustancias (reutiliza crear_mapa de la celda anterior)
mapa_consumo = crear_mapa(df_factores_comuna, "Consumo Sustancias")
if mapa_consumo:
    mapa_consumo.save("mapa_factor_consumo.html")  # se puede abrir en el navegador
    display(mapa_consumo)

Por último, el mapa de la **tasa de intentos 2018–2021 por 100.000 habitantes** (sección 6). Sirve para comparar: si las comunas oscuras en este mapa coinciden con las de los mapas de factores, hay una pista de relación (aunque la sección 11 mostró que es débil). Se guarda como `mapa_tasa_intentos.html`.

*Palmitas, Santa Elena y Altavista tienen pocos casos: su tasa cambia mucho con pocos eventos.*

In [ ]:
# Tercer mapa: tasa de intentos 2018-2021 (tabla `p` de la sección 6)
df_tasa = p[['tasa_intentos']].rename(columns={'tasa_intentos': 'Tasa de intentos'})
mapa_tasa = crear_mapa(df_tasa, "Tasa de intentos",
                       leyenda="Tasa de intentos 2018–2021 (por 100.000 hab. al año)",
                       unidad=" por 100.000 hab.")
if mapa_tasa:
    mapa_tasa.save("mapa_tasa_intentos.html")  # se puede abrir en el navegador
    display(mapa_tasa)

## 13. Comparación de factores entre comunas

Para analizar en detalle cómo se comportan y contrastan todos los factores de riesgo de manera simultánea en el territorio, generaremos un gráfico comparativo multidimensional utilizando subplots estructurados.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar estilo y dimensiones del gráfico comparativo
fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(16, 20))  # sin sharey: cada panel debe tener su propio orden
axes = axes.flatten()

# Lista de factores a graficar
factores = list(df_factores_comuna.columns)

# Paleta de colores consistente
color_principal = "#2f5d8a"

# Iterar sobre cada factor para crear un gráfico de barras individual por comuna
for i, factor in enumerate(factores):
    ax = axes[i]
    # Ordenar los datos de mayor a menor prevalencia para facilitar la comparación
    datos_ordenados = df_factores_comuna[factor].sort_values(ascending=True)

    # Dibujar barras horizontales
    # Comunas con pocos casos en gris y marcadas con *
    colores_barras = ["#b9c0c8" if c in POCOS_CASOS else color_principal for c in datos_ordenados.index]
    bars = ax.barh(datos_ordenados.index.map(marcar), datos_ordenados.values, color=colores_barras, height=0.6, alpha=0.9)

    # Personalización de cada subplot
    ax.set_title(f'Prevalencia de Factor: {factor}', fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('% de Casos', fontsize=10)
    ax.set_xlim(0, max(datos_ordenados.values) * 1.15 if max(datos_ordenados.values) > 0 else 100)
    ax.grid(True, axis='x', linestyle='--', alpha=0.4)

    # Añadir etiquetas de datos al final de cada barra
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.5, bar.get_y() + bar.get_height()/2, f'{width:.1f}%',
                va='center', ha='left', fontsize=8, color='#333333')

# Ocultar el último subplot si sobra espacio (7 factores para 8 posiciones disponibles)
if len(factores) < len(axes):
    fig.delaxes(axes[-1])

plt.suptitle('Comparación de Factores de Riesgo y Desencadenantes de Intentos de Suicidio por Comuna\n(Periodo de Análisis: 2018 - 2022)  ·  * en gris = pocos casos',
             fontsize=16, fontweight='bold', y=0.98, color='#111111')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 14. ¿Qué factores aparecen juntos?

Calcularemos y visualizaremos la correlación de Pearson entre los diferentes factores de riesgo para identificar dinámicas o patrones de coexistencia de problemáticas en las comunas de Medellín.

---
# Conclusiones

## 15. Hallazgo y recomendación

**Hallazgo.** La tasa de intentos de suicidio notificados en Medellín pasó de **88,8 a 103,6 por 100.000 habitantes** entre 2018 y 2022, y la tasa de suicidio de **6,8 a 8,4** entre 2018 y 2021. El problema se concentra en **jóvenes de 10 a 29 años** (72 % de los intentos) y no es parejo en el territorio: **La Candelaria, Robledo, Manrique, Aranjuez y Buenos Aires** tienen a la vez una tasa por encima de la mediana y un crecimiento mayor que el de la ciudad. Además, **44 %** de los casos registrados desde 2018 ya había tenido un intento previo, y las comunas con más intentos también registran más alertas de salud mental en SALVAVIDAS (Spearman 0,75).

**Hallazgos del análisis complementario (secciones 9–14).**
- **Tendencia:** entre 2014 y 2022 los intentos notificados aumentaron en promedio **unos 100 casos por año** (regresión lineal, R² = 0,63). Si esa tendencia se mantiene, Medellín llegaría a **cerca de 2.870 intentos en 2026** (frente a 2.606 en 2022), aunque el rango probable es muy amplio: **entre 2.080 y 3.650**. La dirección (al alza) es más confiable que la cifra exacta.
- **Factores y territorio:** ningún factor explica con claridad por qué unas comunas tienen tasas más altas que otras. Los que más se acercan son los **problemas económicos** (Spearman 0,38) y el **consumo de sustancias** (0,36), pero la asociación es débil y no es significativa (p = 0,12 y 0,14; con la corrección de Holm por probar 7 factores, p > 0,8), con n = 18 comunas. Son **señales a revisar**, no hallazgos confirmados.
- **El trastorno psiquiátrico está en todas partes:** es el antecedente más frecuente (entre 44 % y 68 % de los casos según la comuna), pero no explica por qué unas comunas tienen tasas más altas que otras (Spearman −0,09). Es un factor de toda la ciudad, no de unas pocas comunas.
- **La Candelaria** aparece entre las comunas con más casos asociados a problemas económicos (13,8 %) y consumo de sustancias (23,2 %), lo que es coherente con su lugar entre las prioritarias.

**Recomendación.** Empezar por esas cinco comunas con una ruta de **seguimiento activo después del primer intento**, articulada con colegios, programas de juventud y el sistema SALVAVIDAS, porque casi la mitad de los casos repite y la mayoría son adolescentes y jóvenes. En las comunas con tasas altas, vale la pena **explorar** si la ruta debe incluir apoyo socioeconómico y atención al consumo de sustancias: son los factores con la señal más alta, aunque los datos actuales no alcanzan para confirmarlo. La atención en salud mental, en cambio, debe mantenerse en toda la ciudad, porque el trastorno psiquiátrico está presente en todas las comunas. Si la tendencia al alza continúa, la capacidad de atención tendría que crecer; la cifra exacta es incierta, por lo que conviene actualizar la proyección cada año con los datos nuevos. Como acción de datos, pedir a la Secretaría de Salud mejorar el registro de la comuna, que en 2022 faltó en casi 1 de cada 5 casos.

*Las cifras de este texto se calcularon en las celdas anteriores; si cambian los datos, vuelve a ejecutar el notebook y actualiza el texto.*

## 16. Limitaciones

- Son casos **notificados**: una comuna con más casos puede tener mejor reporte, no necesariamente más intentos.
- La Candelaria es el centro de la ciudad: su tasa puede estar influida por población flotante y habitantes de calle que no aparecen bien en las proyecciones.
- Los porcentajes de factores de Palmitas, Santa Elena y Altavista (marcados con * en las secciones 10 y 13) se basan en pocos casos y pueden variar mucho.
- Factores y mecanismos solo existen desde 2018, y algunos (problemas familiares) están diligenciados en la mitad de los casos.
- En 2022 el 18,6 % de los casos no tiene comuna; por eso el análisis por comuna usa hasta 2021.
- Las tasas por comuna empiezan en 2018, porque las proyecciones de población empiezan ese año.
- Intentos y muertes son bases distintas: se unen por comuna y año, no persona a persona.
- SALVAVIDAS refleja dónde llega el sistema de alertas, no cuántos jóvenes tienen problemas de salud mental; además, cerca de la mitad de sus alertas no tiene comuna.
- La proyección a 2026 es una **línea recta** ajustada con solo 9 años: no considera cambios de política, pandemia ni mejoras en el reporte, y explica alrededor de dos tercios de la variación (R² = 0,63). Su rango probable al 95 % es muy amplio (2.080–3.650 intentos en 2026): debe leerse como una dirección, no como un pronóstico exacto.
- Las correlaciones entre factores y tasas son **ecológicas** (por comuna, no por persona) y con solo 18 comunas: indican asociación, no causa. Con Pearson, económico y consumo parecían significativos (p ≈ 0,04–0,05), pero ese resultado dependía de La Candelaria, cuya tasa es muy superior al resto; con Spearman desaparece.
- La priorización es **exploratoria**: orienta dónde mirar primero, no reemplaza un análisis epidemiológico.